1. Loading of Excel file

In [7]:
import pandas as pd

df = pd.read_excel("../data/raw/All_results_FPKM_Annot.xlsx")



c:\Users\imran\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


In [6]:
%pip install openpyxl



   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


2. Selection of only the columns I need

In [8]:
cols_keep = [
    # Gene annotation
    "Gene_ID", "gene_name", "gene_description", "tf_family", "gene_biotype",
    "kegg_pathway_name", "KOG Description",

    # Raw expression: WT
    "fpkm_WT_avg", "fpkm_WT_stdeva",
    "fpkm_WT_nP_avg", "fpkm_WT_nP_stdeva",

    # Raw expression: pdr2
    "fpkm_pdr2_P_avg", "fpkm_pdr2_P_stdeva",
    "fpkm_pdr2_nP_avg", "fpkm_pdr2_nP_stdeva",

    # Raw expression: l1l2
    "fpkm_l1l2_P_avg", "fpkm_l1l2_P_stdeva",
    "fpkm_l1l2_nP_avg", "fpkm_l1l2_nP_stdeva",

    # WT tissues
    "fpkm_WT_S_avg", "fpkm_WT_S_stdeva",
    "fpkm_WT_R_avg", "fpkm_WT_R_stdeva",

    # Differential expression: WT
    "fpkm_WT_nPvsWT_P_log2foldchange",
    "fpkm_WT_nPvsWT_P_pvalue",
    "fpkm_WT_nPvsWT_P_padjust",

    # Differential expression: pdr2
    "fpkm_pdr2_nPvspdr2_P_log2foldchange",
    "fpkm_pdr2_nPvspdr2_P_pvalue",
    "fpkm_pdr2_nPvspdr2_P_padjust",

    # Differential expression: l1l2
    "fpkm_l1l2_nPvsl1l2_P_log2foldchange",
    "fpkm_l1l2_nPvsl1l2_P_pvalue",
    "fpkm_l1l2_nPvsl1l2_P_padjust",

    # Genotype comparisons
    "fpkm_l1l2_PvsWT_P_log2foldchange",
    "fpkm_l1l2_PvsWT_P_pvalue",
    "fpkm_l1l2_PvsWT_P_padjust",

    "fpkm_pdr2_PvsWT_P_log2foldchange",
    "fpkm_pdr2_PvsWT_P_pvalue",
    "fpkm_pdr2_PvsWT_P_padjust",

    "fpkm_l1l2_Pvspdr2_P_log2foldchange",
    "fpkm_l1l2_Pvspdr2_P_pvalue",
    "fpkm_l1l2_Pvspdr2_P_padjust",

    # Root vs shoot
    "fpkm_WT_SvsWT_R_log2foldchange",
    "fpkm_WT_SvsWT_R_pvalue",
    "fpkm_WT_SvsWT_R_padjust",
]

df = df[cols_keep]


3. Renaming of columns

In [9]:
df = df.rename(columns={
    "fpkm_WT_avg": "WT_P",
    "fpkm_WT_nP_avg": "WT_nP",

    "fpkm_pdr2_P_avg": "pdr2_P",
    "fpkm_pdr2_nP_avg": "pdr2_nP",

    "fpkm_l1l2_P_avg": "l1l2_P",
    "fpkm_l1l2_nP_avg": "l1l2_nP",

    "fpkm_WT_nPvsWT_P_log2foldchange": "WT_log2FC",
    "fpkm_WT_nPvsWT_P_pvalue": "WT_pvalue",
    "fpkm_WT_nPvsWT_P_padjust": "WT_padjust",

    "fpkm_pdr2_nPvspdr2_P_log2foldchange": "pdr2_log2FC",
    "fpkm_pdr2_nPvspdr2_P_pvalue": "pdr2_pvalue",
    "fpkm_pdr2_nPvspdr2_P_padjust": "pdr2_padjust",

    "fpkm_l1l2_nPvsl1l2_P_log2foldchange": "l1l2_log2FC",
    "fpkm_l1l2_nPvsl1l2_P_pvalue": "l1l2_pvalue",
    "fpkm_l1l2_nPvsl1l2_P_padjust": "l1l2_padjust",
})


4. Conversion of numeric columns

In [10]:
numeric_cols = df.columns.drop(["Gene_ID", "gene_name", "gene_description"])
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")


5. Convert wide expression data into long format

In [11]:
expr_cols = ["WT_P", "WT_nP", "pdr2_P", "pdr2_nP", "l1l2_P", "l1l2_nP"]

df_expr = df.melt(
    id_vars=["Gene_ID", "gene_name"],
    value_vars=expr_cols,
    var_name="condition",
    value_name="expression"
)


6. Export cleaned tables

In [12]:
df_expr.to_csv("../data/processed/expression.csv", index=False)
df.to_csv("../data/processed/foldchange.csv", index=False)


7. Generation of annotation csv file. I will need it for later merger

In [15]:
df = pd.read_excel("../data/raw/All_results_FPKM_Annot.xlsx", engine="openpyxl")

annot_cols = [
    "Gene_ID", "gene_name", "gene_description", "tf_family", "gene_biotype",
    "kegg_pathway_name", "KOG Description", "range", "Range%", "sum", "Avrg", "express"
]

df_annot = df[annot_cols]
df_annot.to_csv("../data/processed/annotation.csv", index=False)



c:\Users\imran\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
